# Unit 9: Translation (German → English)

[Qwen2.5-VL](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct) is a
vision-language model, but it is also a strong **text** chat model. Here we use
it **text-only** (no images) to translate the German transcript from
`00_PrepData` into English.

We translate **segment by segment** so each request stays short and the
timestamps are preserved, this also scales to clips of any length.

**Sources**
- Qwen2.5-VL: <https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct> ([report](https://arxiv.org/abs/2502.13923))
- Model class `QwenChat` lives in `src/video_pipeline/qwen_text.py`.

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

# Make the vendored `video_pipeline` package importable even if the project
# was not installed (e.g. running `jupyter` outside `uv run`).
SRC = Path.cwd() / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PREPARED = Path("prepared")
PREPARED.mkdir(exist_ok=True)

transcript = json.loads((PREPARED / "transcript.json").read_text(encoding="utf-8"))
print(f"loaded transcript: {len(transcript['segments'])} segments, "
      f"{len(transcript['text'])} chars")
print("\nfirst 300 chars (German):\n", transcript["text"][:300])

loaded transcript: 5 segments, 1349 chars

first 300 chars (German):
 Ohne den Golfstrom sähe das Klima in Europa ganz anders aus. Karge Landschaften statt blumiger Wiesen. Der Meeresstrom liefert tagtäglich gigantische Mengen an Wärmeenergie. Ein konstant wehender Wind, der südöstliche Passatwind, weht warmes Wasser vom Äquator in den Golf von Mexiko. Dort entsteht d


## Translate each segment with Qwen2.5-VL

`QwenChat.translate(text, source, target)` wraps a single system+user chat turn.
We load the model once (`with QwenChat() as qc:`) and reuse it across all
segments, then it is unloaded and the GPU cache is cleared on exit.

In [2]:
from video_pipeline.qwen_text import QwenChat

translated_segments = []
with QwenChat() as qc:
    for i, seg in enumerate(transcript["segments"], start=1):
        english = qc.translate(seg["text"], source="German", target="English")
        translated_segments.append({**seg, "text": english})
        print(f"[{i}/{len(transcript['segments'])}] {english[:70]}")

english_text = "\n".join(s["text"] for s in translated_segments)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

/home/azureuser/localfiles/jonas_g/GAI4/Unit_9/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

/home/azureuser/localfiles/jonas_g/GAI4/Unit_9/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1/5] Without the Gulf Stream, the climate in Europe would look very differe
[2/5] Scarcer landscapes instead of flower-rich meadows. The ocean current p
[3/5] From here, the water masses flow deep into the north, rise to the uppe
[4/5] Water layers rise and flow around Africa back into the Atlantic.
[5/5] The Gulf Stream is thus part of a global flow system.


## Save a `TranslationResult`

We store the full English text in the project's `TranslationResult` schema so
the next notebook (and the CLI) can consume it.

In [3]:
from video_pipeline.models import TranslationResult

result = TranslationResult(
    source_path=str(PREPARED / "transcript.json"),
    source_language="German",
    target_language="English",
    source_text=transcript["text"],
    text=english_text,
)
out = PREPARED / "translation.json"
out.write_text(result.model_dump_json(indent=2), encoding="utf-8")
print("wrote", out, "-", len(english_text), "chars")

wrote prepared/translation.json - 1279 chars


In [4]:
# German vs English, side by side per segment.
for de, en in zip(transcript["segments"][:6], translated_segments[:6]):
    print(f"[{de['start']:6.1f}s] DE: {de['text']}")
    print(f"          EN: {en['text']}\n")

[   0.0s] DE: Ohne den Golfstrom sähe das Klima in Europa ganz anders aus.
          EN: Without the Gulf Stream, the climate in Europe would look very different.

[   5.0s] DE: Karge Landschaften statt blumiger Wiesen. Der Meeresstrom liefert tagtäglich gigantische Mengen an Wärmeenergie. Ein konstant wehender Wind, der südöstliche Passatwind, weht warmes Wasser vom Äquator in den Golf von Mexiko. Dort entsteht der Golfstrom. Die Drehung der Erde und die Westwinde schicken den Golfstrom Richtung Europa. Vor der Küste Europas spaltet sich der Golfstrom. Ein Teilstrom fließt weiter hoch in den Norden Europas. Deshalb heißt er jetzt nordatlantischer Strom. Er wirkt dabei wie eine riesige Heizung. Sein warmes Wasser erwärmt die Luft und sorgt somit für ein mildes Klima. Auf dem Weg in den Norden verdunstet ein Teil des warmen Wassers. Dadurch steigt der Salzgehalt. Außerdem kühlt der nordatlantische Strom immer mehr ab. Vor der Küste Grönlands sinkt das Wasser ab, weil kaltes und salziges

---
**Next:** `02_Summarize_OCR.ipynb` — summarise the English transcript and OCR the
on-screen text from the video frames.